[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carrycooldude/ModelGarden-QNN-LiteRT/blob/main/google_colab/LiteRT_Gemma4_NPU_AOT_Compilation.ipynb)

# Gemma 4 E2B - LiteRT NPU AOT Compilation

This notebook performs **Ahead-of-Time (AOT) compilation** for the Gemma 4 E2B model, targeting the **Qualcomm Snapdragon 8 Elite (SM8750)** NPU.

### Why use Colab?
The LiteRT AOT compiler (`apply_plugin_main`) is a native x86_64 Linux binary. Running this on ARM64 Windows or WSL requires emulation. Colab provides a native x86_64 Linux environment, making it the most reliable way to generate the required `TF_LITE_AUX` payload.

## Step 1: Install Dependencies
Install the Google AI Edge LiteRT package.

In [ ]:
!pip install ai-edge-litert

## Step 2: Upload Your Model and SDK

You need to upload:
1. **`gemma-4-E2B-it.litertlm`**: The original model file.
2. **`qairt_sdk.zip`**: The Qualcomm AI Runtime SDK (v2.42.0 or newer).

*Note: Uploading 2.5GB+ via the browser can be slow. It is recommended to mount Google Drive or use `wget` if you have a direct link.*

In [ ]:
from google.colab import files
import os

if not os.path.exists("gemma-4-E2B-it.litertlm"):
    print("Please upload gemma-4-E2B-it.litertlm")
    # uploaded = files.upload()

if not os.path.exists("qairt_sdk.zip"):
    print("Downloading QAIRT SDK...")
    # Using the v2.42 link based on your local path
    !wget "https://softwarecenter.qualcomm.com/api/download/software/sdks/Qualcomm_AI_Runtime_Community/All/2.42.0.251225/v2.42.0.251225.zip" -O qairt_sdk.zip

## Step 3: Extract SDK and Set Environment Variables
We need to point the compiler to the Qualcomm SDK binaries.

In [ ]:
!unzip -q qairt_sdk.zip -d qairt_sdk

import os
os.environ["QAIRT_ROOT"] = "/content/qairt_sdk/qairt/2.42.0.251225"
# Use x86_64-linux-clang for Colab (Native execution)
os.environ["PATH"] = os.environ["QAIRT_ROOT"] + "/bin/x86_64-linux-clang:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = os.environ["QAIRT_ROOT"] + "/lib/x86_64-linux-clang:" + os.environ.get("LD_LIBRARY_PATH", "")

print(f"QAIRT_ROOT set to: {os.environ['QAIRT_ROOT']}")

## Step 4: Run AOT Compilation
This is the core compilation step. It targets the **SM8750** (Snapdragon 8 Elite).

In [ ]:
from ai_edge_litert.aot import aot_compile as aot_lib
from ai_edge_litert.aot.vendors.qualcomm import target as qnn_target

model_path = "gemma-4-E2B-it.litertlm"
out_dir = "compiled_output"
os.makedirs(out_dir, exist_ok=True)

print(f"Starting AOT compilation for {model_path}...")
sm8750_target = qnn_target.Target(qnn_target.SocModel.SM8750)

try:
    compiled_models = aot_lib.aot_compile(
        model_path, 
        output_dir=out_dir,
        target=[sm8750_target]
    )
    print(f"SUCCESS! Compiled model is in: {out_dir}/")
except Exception as e:
    print(f"COMPILATION FAILED: {e}")

## Step 5: Download the Result
The resulting model now contains the hardware-specific `TF_LITE_AUX` block.

In [ ]:
import glob
from google.colab import files

compiled_files = glob.glob(f"{out_dir}/*.litertlm")
if compiled_files:
    print(f"Downloading {compiled_files[0]}...")
    files.download(compiled_files[0])
else:
    print("No compiled model found.")